# Zephyr Weather Model - Google Colab Training

Train the PatchTST weather forecasting model on a **free GPU** using Google Colab.

## Quick Start
1. Open this notebook in Colab: **Runtime → Change runtime type → T4 GPU**
2. Run all cells (**Runtime → Run all**)
3. When prompted, upload your `data.csv` file
4. Training takes ~30 minutes on a T4 GPU
5. Download the trained model when complete

## 1. Setup Environment

In [ ]:
# Install tsai (includes PyTorch, FastAI, and time series tools)
# This takes 2-3 minutes on a fresh Colab runtime
!pip install -q tsai>=0.4.1

In [ ]:
# Verify GPU is available
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → T4 GPU")
    print("Training will be much slower on CPU.")

## 2. Upload Training Data

Choose one of two methods to load your `data.csv`:
- **Option A**: Direct upload (simplest, good for files < 100MB)
- **Option B**: Google Drive (persistent across sessions)

In [ ]:
# === OPTION A: Direct upload (recommended for first time) ===
# A file picker will appear — select your data.csv file

from google.colab import files
import os

if not os.path.exists("data.csv"):
    print("Select your data.csv file to upload...")
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print("data.csv already present, skipping upload.")

In [ ]:
# === OPTION B: Google Drive (uncomment to use) ===
# Keeps your data between sessions so you don't need to re-upload.
#
# 1. Upload data.csv to your Google Drive first
# 2. Update the path below to match your Drive location

# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drive/MyDrive/zephyr/data.csv ./data.csv

## 3. Load and Inspect Data

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Stations: {df['id'].nunique()}")
print(f"\nSample rows:")
df.head()

## 4. Training Configuration

Adjust these settings before training. The defaults work well for the Coronet Peak dataset.

In [ ]:
# --- Training parameters ---
WINDOW_LEN = 60       # Input: 60 timesteps = 10 hours of history
HORIZON = 6           # Output: 6 timesteps = 1 hour prediction
VALID_SIZE = 0.2      # 20% validation split
N_EPOCHS = 20         # Training epochs
LR_MAX = 1e-3         # Max learning rate (one-cycle schedule)
BATCH_SIZE = 128      # Increase to 512 or 1024 if GPU memory allows
ARCH = "PatchTST"     # Model architecture
RANDOM_STATE = 23     # Random seed

# Weather variables to predict
FEATURES = ["temperature", "wind_average", "wind_gust", "wind_bearing"]

# Model output
MODEL_NAME = "weather_forecast_multi_station"

print("Configuration set.")

## 5. Prepare Data

- Extract cyclical temporal features (hour, day-of-year, weekday)
- Handle missing values per station (forward/backward fill)
- Create sliding windows for the time series model

In [ ]:
from tsai.all import (
    Nan2Value,
    ShowGraph,
    SlidingWindowPanel,
    TSForecaster,
    TSStandardize,
    get_splits,
    mae,
    rmse,
)


def extract_temporal_features(df, timestamp_col="timestamp"):
    """Create cyclical sin/cos features from timestamp."""
    df = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df[timestamp_col]):
        df[timestamp_col] = pd.to_datetime(df[timestamp_col], unit="s")

    dt = df[timestamp_col]
    hour_of_day = dt.dt.hour + dt.dt.minute / 60
    df["hour_sin"] = np.sin(2 * np.pi * hour_of_day / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour_of_day / 24)

    day_of_year = dt.dt.dayofyear
    df["day_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    df["day_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)

    day_of_week = dt.dt.dayofweek
    df["weekday_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    return df


TEMPORAL_FEATURES = ["hour_sin", "hour_cos", "day_sin", "day_cos", "weekday_sin", "weekday_cos"]

# Sort and clean
df_sorted = df.sort_values(["id", "timestamp"]).reset_index(drop=True)
df_sorted = extract_temporal_features(df_sorted)
df_sorted[FEATURES] = df_sorted.groupby("id")[FEATURES].transform(
    lambda x: x.ffill().bfill()
)
df_sorted = df_sorted.dropna(subset=FEATURES)

all_features = FEATURES + TEMPORAL_FEATURES

print(f"Stations: {df_sorted['id'].nunique()}")
print(f"Rows: {len(df_sorted):,}")
print(f"Features: {all_features}")
print(f"Missing values: {df_sorted[FEATURES].isnull().sum().sum()}")

In [ ]:
# Create sliding windows
X, y = SlidingWindowPanel(
    window_len=WINDOW_LEN,
    horizon=HORIZON,
    unique_id_cols=["id"],
    get_x=all_features,
    get_y=all_features,
    sort_by="timestamp",
)(df_sorted)

print(f"X shape: {X.shape}  (samples, timesteps, features)")
print(f"y shape: {y.shape}  (samples, horizon, features)")
assert not np.isnan(X).any(), "NaN in X!"
assert not np.isnan(y).any(), "NaN in y!"
print("Data quality: OK")

In [ ]:
# Train/validation split
splits = get_splits(
    range(len(y)), valid_size=VALID_SIZE, shuffle=True, random_state=RANDOM_STATE
)
print(f"Train: {len(splits[0]):,} samples")
print(f"Valid: {len(splits[1]):,} samples")

## 6. Train Model

PatchTST with one-cycle learning rate schedule. ~30 minutes on T4 GPU.

In [ ]:
import time

fcst = TSForecaster(
    X,
    y,
    splits=splits,
    path="models",
    batch_tfms=[TSStandardize(), Nan2Value()],
    bs=BATCH_SIZE,
    arch=ARCH,
    metrics=[mae, rmse],
    cbs=ShowGraph(),
)

print(f"Training {ARCH} for {N_EPOCHS} epochs...")
start = time.time()
fcst.fit_one_cycle(n_epoch=N_EPOCHS, lr_max=LR_MAX)
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")

## 7. Evaluate

In [ ]:
# Validation metrics
preds, targets = fcst.get_X_preds(X[splits[1]], y[splits[1]])

mae_per_var = np.abs(preds - targets).mean(axis=(0, 1))

print("Validation MAE per variable:")
print("-" * 35)
for feature, mae_val in zip(FEATURES, mae_per_var):
    print(f"  {feature:20s}  {mae_val:.4f}")

print(f"\nFinal train loss: {fcst.recorder.values[-1][0]:.4f}")
print(f"Final valid loss: {fcst.recorder.values[-1][1]:.4f}")

## 8. Save and Download Model

In [ ]:
import os

# Save model
model_filename = f"{MODEL_NAME}.pkl"
fcst.export(f"models/{model_filename}")
model_path = f"models/{model_filename}"
model_size = os.path.getsize(model_path) / 1e6
print(f"Model saved: {model_path} ({model_size:.1f} MB)")

In [ ]:
# Download to your local machine
from google.colab import files
files.download(model_path)
print(f"Downloading {model_filename}...")

In [ ]:
# === Optional: Save to Google Drive instead ===
# from google.colab import drive
# drive.mount("/content/drive")
# !cp {model_path} /content/drive/MyDrive/zephyr/{model_filename}
# print(f"Model copied to Google Drive: zephyr/{model_filename}")